# Pokémon LLM Agent with Unsloth on AMD ROCm

**Authors:** Yue Yuan ([yueyuan](https://github.com/yueyuan), yueyuan@amd.com), Bill He ([billishyahao](https://github.com/billishyahao), bill.he@amd.com)

**Knowledge level:** Intermediate

**Phase 13 topic:** *Pokémon LLM Agent with Unsloth.* This walkthrough is **step by step**: learn **Pokémon Showdown** rules with an interactive demo and battle visuals, compare **published checkpoints** on the same prompt, walk through **dataset processing** (long replay logs → short training rows), fine-tune **Qwen3-4B** with **Unsloth** + **TRL** on **AMD ROCm**, then use **GRPO** with a carefully designed **reward function** so the model learns valid `move …` / `switch …` actions from **raw replay logs**. A **mini-eval** matches the **`eval_showdown_agent.py`** protocol from the [companion repository](#companion-repository-eval-utilities-published-separately) (cloned in Step 0).

**Published reference weights (optional):** [Hugging Face model repository](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-v6) — same recipe at production scale; this tutorial stays intentionally small.

**Training data (streaming):** [`milkkarten/pokemon-showdown-replays-merged`](https://huggingface.co/datasets/milkkarten/pokemon-showdown-replays-merged) (Apache-2.0)

---

## Prerequisites

This tutorial was developed and tested using the setup below.

### Operating system

* **Ubuntu 22.04 or 24.04** — match the ROCm/PyTorch container you use (recommended path below).

### Hardware

* **AMD Instinct™ GPUs**: Tested on **AMD Instinct MI300**. Use an AMD Instinct or other ROCm-supported GPU per [AMD system requirements](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html).
* **VRAM**: Comfortable headroom for **Qwen3-4B** in **bfloat16** with LoRA is typically **≥ 24 GB**.
* Several **GB of free disk** for HF cache and checkpoints (exact size depends on cache location).

### Software

* **ROCm aligned with your PyTorch build** — Prefer an official **AMD `rocm/pytorch`** image where **PyTorch is already compiled for ROCm (HIP)**. Do **not** rely on plain **PyTorch** wheels from PyPI defaults inside a ROCm-backed machine unless you install from the [**official PyTorch ROCm wheel index**](https://pytorch.org/get-started/locally/) matched to your ROCm install; CPU-only builds do not expose HIP and Unsloth will fail device detection.
  After ROCm/Docker setup, verify GPUs with:

    ```bash
    amd-smi
    ```

    For ROCm 6.4 and earlier, some environments use **`rocm-smi`** instead of **`amd-smi`**.

* **Python 3.10+** — Use a Python release supported by your ROCm PyTorch image and Unsloth.

* **Docker** *(optional local path)* — If you reproduce outside AMD Developer Cloud, install Docker per your OS. For non-root access:

    ```bash
    sudo usermod -aG docker $USER
    newgrp docker
    ```

    Verify with `docker run hello-world`.

### Accounts and tokens

* A Hugging Face account is optional for *reading* public models/datasets; if you hit limits, run `huggingface-cli login` or set `HF_TOKEN`.
* Publishing checkpoints ([Step 11](#8-export-and-publish)) needs `hf` CLI auth ([Hub CLI docs](https://huggingface.co/docs/huggingface_hub/guides/cli)).

<a id="companion-repository-eval-utilities-published-separately"></a>

### Companion repository (eval utilities, published separately)

The **AMD AI Developer Hub HTML** renders this notebook **without** a `scripts/` tree next to it. Evaluation helpers (`showdown_agent_eval.py`, CLI `eval_showdown_agent.py`) live in the companion Git repo **`pokemon-rocm-tutorial`** (maintainers: keep the canonical URL synchronized with wherever you publish that repo).

- By default Step 0 clones **`https://github.com/ROCm/pokemon-rocm-tutorial.git`** into **`./pokemon-rocm-tutorial-extras`** (`--depth 1`). Replace the URL with your **published** fork until the ROCm org mirror exists, using **`POKEMON_ROCM_EXTRAS_REPO`** (and optionally **`POKEMON_ROCM_EXTRAS_DIR`**).

<a id="rocm-troubleshooting"></a>

### ROCm HIP / PyTorch troubleshooting (Developer Cloud & local)

If **`import unsloth`** raises **`NotImplementedError`** mentioning *“AMD ROCm GPU … no usable HIP accelerator”* (`unsloth_zoo.device_type`), your interpreter is almost certainly running **CPU PyTorch or a mismatched CPU wheel**, even though the host has ROCm drivers.

Fix (pick **one**, consistent with your ROCm stack):

1. **Prefer the ROCm Docker image** in [Prepare the tutorial environment](#prepare-the-tutorial-environment) so Torch + ROCm match out of the box.
2. If you must repair an existing env, reinstall **ROCm Torch wheels** using the [**official PyTorch ROCm wheel index**](https://pytorch.org/get-started/locally/). For example:

    ```bash
    pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm<your-rocm>/
    ```

    Unsloth’s checks may also surface [`uv pip …`](https://github.com/unslothai/unsloth/issues/3520) guidance; **`UV_SKIP_WHEEL_FILENAME_CHECK=1`** applies only when `uv` refuses certain wheel filenames.

3. After reinstall, **`python -c "import torch; print(torch.__version__, torch.version.hip, torch.cuda.is_available())"`** should show a **HIP version string** and **`torch.cuda.is_available() == True`** on ROCm builds.

**Unsloth / numpy stack:** If `pip install` reports resolver conflicts (**torch**, **transformers**, **trl**, **peft**, …), align versions with **`pip install unsloth` / `requirements.txt`** of the companion repo **instead of forcing partially satisfied `--no-deps` installs**.

---

<a id="prepare-the-tutorial-environment"></a>

## Prepare the tutorial environment

Follow these steps to match a current **ROCm PyTorch** runtime on **Ubuntu 22.04 or 24.04**.

### 1. Pull the ROCm PyTorch Docker image

Confirm [system requirements](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html), then pull the **latest** matching **`rocm/pytorch`** image from [Docker Hub tags](https://hub.docker.com/r/rocm/pytorch/tags):

```bash
docker pull rocm/pytorch:latest  # or the newest tag that matches your GPU + ROCm install
```

### 2. Launch the container

Expose the GPUs and shared memory similarly to other ROCm Jupyter tutorials ([reference pattern](https://github.com/ROCm/gpuaidev-internal/blob/main/docs/notebooks/inference/build_airbnb_agent_mcp.ipynb)):

```bash
docker run -it --rm \
  --network=host \
  --device=/dev/kfd \
  --device=/dev/dri \
  --group-add=video \
  --ipc=host \
  --cap-add=SYS_PTRACE \
  --security-opt seccomp=unconfined \
  --shm-size 8G \
  -v $(pwd):/workspace \
  -w /workspace/notebooks \
  rocm/pytorch:latest
```

Mount the folder that will hold this notebook (`/workspace/notebooks` in the example matches the Airbnb layout; AMD Developer Cloud Jupyter usually provisions this for you—in that case you only need a **Torch+ROCm** kernel on your target GPU, not necessarily this exact `docker run`). If **`python -c "import torch; print(torch.version.hip)"`** prints **`None`** while `amd-smi` shows GPUs on the host, your kernel lacks HIP-enabled PyTorch; switch images or reinstall per **[ROCm HIP / PyTorch troubleshooting](#rocm-troubleshooting)**.

Download this notebook from the [AI Developer Hub sources](https://github.com/ROCm/gpuaidev) (published HTML renders under [AMD AI Developer Hub](https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/)); shallow-cloning the companion **`pokemon-rocm-tutorial`** repo is automated in Step 0.

### 3. Install Jupyter (if absent) and launch

Inside the container (when not using a hosted Jupyter service):

```bash
pip install jupyter
jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

Use a free port if **8888** is taken.

### Before you run cells

1. **Kernel**: Python from the ROCm Torch environment above.
2. **Run top to bottom** the first time; after dependency installs **restart once** then **Run All** so Unsloth import order stays consistent.

---

## What you will learn

- **Step by step:** Pokémon Showdown rules, protocol lines, and an interactive battle demo (with [PokéAPI](https://pokeapi.co/) + optional live Showdown matching).
- **Compare checkpoints:** run the same battle prompt on the base model and published Hugging Face weights.
- **Dataset processing:** see how Python turns a **long** replay log into a **short** supervised row.
- Configure **HF cache**, validate **Unsloth / TRL / ROCm**, attach **LoRA**, and run a short **SFT** alignment pass.
- **GRPO** with a multi-signal **reward function** (format, legality, type match).
- **Inference**, `|request|`-aware legality checks, and **mini-eval** via **`showdown_agent_eval`**.

---

## Tutorial outline (step by step)

| Step | Section | Purpose |
|------|---------|---------|
| 0 | Environment | Writable cache, companion clone, ROCm env vars |
| 1 | Install runtime | Idempotent pip installs |
| 2 | Validate ROCm | Imports, GPU visibility |
| 3 | **Learn Showdown rules** | Battle images, protocol primer, interactive demo |
| 4 | **Compare model weights** | Same prompt on base vs published checkpoints |
| 5 | **Dataset processing** | Long replay log → short chat row (hands-on) |
| 6 | Model + LoRA | `Qwen/Qwen3-4B` + PEFT |
| 7 | Short SFT | Alignment demo before GRPO |
| 8 | **GRPO + reward design** | Policy optimization with format + legality rewards |
| 9 | Inference + legality | `showdown_agent_eval` helpers |
| 10 | Mini-eval + compare | Metrics + before/after checkpoint table |
| 11 | Export / Hub | Optional merge & upload |

---

## Design notes (robustness on AMD systems)

1. Dependency installation runs **before** heavy imports.
2. Cache and temp paths **fall back** if `/shared-docker` is unavailable.
3. Data preparation uses **`streaming=True`** end to end — no full 40GB+ local extract.
4. **`load_in_4bit=False`** and **`adamw_torch`** in the tutorial path for fewer ROCm surprises than 4-bit + 8-bit optimizers.
5. Notebook outputs are **cleared** in source so readers see only their own runs.

---

## Post-publication maintenance (for primary authors)

When submitting to the **AMD AI Developer Hub** / `ROCm/gpuaidev-internal`: **ship only this `.ipynb`** in the docs tree (no `scripts/`); keep `showdown_agent_eval.py` in **`pokemon-rocm-tutorial`** synchronized with the URLs in the intro. Re-run the notebook on the **declared ROCm image and GPU SKU** after each major bump; refresh **Prerequisites** if Python, PyTorch, or Unsloth minimums change. **Primary authors:** Yue Yuan (yueyuan@amd.com), Bill He (bill.he@amd.com).

If you extend training (more steps), document expected VRAM and runtime so reviewers can reproduce.


In [ ]:
# Step 0 — Environment: ROCm/HIP-friendly defaults, writable HF cache, and companion eval helpers.
# Run this first on the target GPU machine; adjust HIP_VISIBLE_DEVICES if you use multiple GPUs.
import os
import platform
import shutil
import subprocess
from pathlib import Path


def pick_writable_dir(candidates):
    for candidate in candidates:
        path = Path(candidate)
        try:
            path.mkdir(parents=True, exist_ok=True)
            probe = path / ".cursor_write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return path
        except Exception:
            continue
    raise RuntimeError(f"No writable directory found in: {candidates}")


def ensure_pokemon_eval_repo_root() -> Path:
    """Return repo root containing scripts/eval/showdown_agent_eval.py (clone shallow if needed)."""
    cwd = Path.cwd().resolve()
    extras_repo = os.environ.get(
        "POKEMON_ROCM_EXTRAS_REPO",
        "https://github.com/ROCm/pokemon-rocm-tutorial.git",
    )
    extras_dir = Path(
        os.environ.get("POKEMON_ROCM_EXTRAS_DIR", cwd / "pokemon-rocm-tutorial-extras"),
    ).resolve()

    eval_rel = Path("scripts") / "eval" / "showdown_agent_eval.py"

    def find_root():
        for base in (cwd, *cwd.parents, extras_dir):
            if (base / eval_rel).is_file():
                return base.resolve()
        return None

    found = find_root()
    if found is not None:
        os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"] = str(found)
        print(f"Pokémon showdown eval helpers: {found / Path('scripts') / 'eval'}")
        return found

    git = shutil.which("git")
    if not git:
        raise RuntimeError(
            "`git` is required to shallow-clone eval utilities unless you opened this notebook "
            "from inside a pokemon-rocm-tutorial checkout (with scripts/eval alongside this file)."
        )

    if extras_dir.exists():
        has_eval = (extras_dir / eval_rel).is_file()
        if has_eval:
            found = extras_dir.resolve()
            os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"] = str(found)
            print(f"Pokémon showdown eval helpers (existing tree): {found / Path('scripts') / 'eval'}")
            return found
        if not any(extras_dir.iterdir()):
            extras_dir.rmdir()
        else:
            missing = extras_dir / eval_rel
            raise RuntimeError(
                f"{extras_dir} exists but is missing expected file {missing} — "
                "remove/rename it, or set POKEMON_ROCM_EXTRAS_REPO / POKEMON_ROCM_EXTRAS_DIR."
            )

    extras_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [git, "clone", "--depth", "1", extras_repo, str(extras_dir)],
        check=True,
    )

    if not (extras_dir / eval_rel).is_file():
        raise RuntimeError(f"Clone finished but showdown_agent_eval.py not found under {extras_dir}")

    found = extras_dir.resolve()
    os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"] = str(found)
    print(f"Cloned companion Pokémon eval repo into {found}")
    return found


cache_root = pick_writable_dir([
    "/shared-docker/.cache/huggingface",
    "/data/huggingface",
    "/tmp/pokemon-hf-cache",
])
tmp_root = pick_writable_dir([
    "/shared-docker/.cache/tmp",
    "/data/tmp",
    "/tmp/pokemon-tmp",
])

os.environ.setdefault("HIP_VISIBLE_DEVICES", "0")
os.environ.setdefault("ROCR_VISIBLE_DEVICES", os.environ["HIP_VISIBLE_DEVICES"])
os.environ.setdefault("TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL", "1")
os.environ.setdefault("PYTORCH_HIP_ALLOC_CONF", "expandable_segments:False")
os.environ.setdefault("UNSLOTH_SKIP_TORCHVISION_CHECK", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

os.environ["HF_HOME"] = str(cache_root)
os.environ["HF_DATASETS_CACHE"] = str(cache_root / "datasets")
os.environ["TMPDIR"] = str(tmp_root)

Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)

ensure_pokemon_eval_repo_root()

cache_usage = shutil.disk_usage(cache_root)
print(f"platform={platform.platform()}")
print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE={os.environ['HF_DATASETS_CACHE']}")
print(f"TMPDIR={os.environ['TMPDIR']}")
print(f"HIP_VISIBLE_DEVICES={os.environ['HIP_VISIBLE_DEVICES']}")
print(f"free_space_gb={cache_usage.free / (1024 ** 3):.1f}")

## 1. Install the runtime stack
**Tutorial content — dependencies.** This cell uses only the Python standard library before any ML imports. It detects missing packages, prints versions already installed, and runs `pip install` only for gaps.

On a prebuilt **AMD ROCm** image, keeping the image’s pinned PyTorch + ROCm stack and adding Unsloth on top is usually safer than forcing broad downgrades inside the notebook.

If `pip` reports hard dependency conflicts (**torch**, **transformers**, **trl**, …), install from **`$POKEMON_SHOWDOWN_EVAL_ROOT/requirements.txt`** (the companion checkout from Step 0) before retrying **`unsloth`**, rather than chaining partially satisfied **`--no-deps`** installs unless you intentionally pin everything yourself.

**After installing:** restart the Jupyter kernel once, then **Run All** from the top so import order (Unsloth first) stays correct.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

core_runtime = [
    "transformers",
    "trl",
    "datasets",
    "tokenizers",
    "huggingface_hub",
    "accelerate",
    "peft",
    "psutil",
    "sentencepiece",
    "protobuf",
    "tyro",
    "hf_transfer",
    "einops",
]
required_modules = [
    "unsloth",
    "unsloth_zoo",
    "transformers",
    "trl",
    "datasets",
    "tokenizers",
    "huggingface_hub",
    "accelerate",
    "peft",
    "psutil",
    "sentencepiece",
    "google.protobuf",
    "tyro",
    "einops",
]

missing = [name for name in required_modules if importlib.util.find_spec(name) is None]

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--no-deps",
        "unsloth",
        "unsloth_zoo",
    ])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *core_runtime,
    ])
    print("Install completed. Rerun the notebook from the top if this is a fresh environment.")
else:
    print("All required packages are already available.")

print("Detected package versions:")
for package_name in ["unsloth", "transformers", "trl", "datasets", "tokenizers", "huggingface_hub"]:
    try:
        print(f"  {package_name}={importlib.metadata.version(package_name)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {package_name}=missing")

## 2. Validate the AMD ROCm runtime
**Tutorial content — hardware smoke test.** The next cell imports in **Unsloth-first** order, prints package versions, and checks that **PyTorch** reports a GPU (ROCm builds typically expose HIP via the CUDA-like API). If this fails, fix the container or driver stack before training — the AMD AI Developer Hub review expects this step to pass on the declared SKU.

In [ ]:
import unsloth
import datasets
import huggingface_hub
import torch
import transformers
import trl


datasets.config.HF_DATASETS_CACHE = os.environ["HF_DATASETS_CACHE"]

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"trl={trl.__version__}")
print(f"datasets={datasets.__version__}")
print(f"huggingface_hub={huggingface_hub.__version__}")
print(f"unsloth={getattr(unsloth, '__version__', 'unknown')}")
print(f"hip_version={getattr(torch.version, 'hip', None)}")

if not torch.cuda.is_available():
    raise RuntimeError("No ROCm-visible GPU found. This notebook expects an AMD GPU environment.")

print(f"gpu_count={torch.cuda.device_count()}")
print(f"gpu_name={torch.cuda.get_device_name(0)}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")

<a id="step-3-learn-showdown-rules"></a>

## 3. Learn Pokémon Showdown rules (step by step)

Before training, get comfortable with what the model must output. A Showdown singles turn is always **one line**:

- `move <MoveName>` — use a move on the active Pokémon (optional trailing `terastallize`)
- `switch <Species>` — send in a benched Pokémon

The server speaks in **protocol lines** (`|turn|`, `|move|`, `|switch|`, `|request|{...}`). Our agent reads the log prefix and prints the next command.

### 3a. Visual primer — what a turn looks like

| Concept | What you see in battle | Protocol echo |
|---------|------------------------|---------------|
| Active vs bench | Front sprite vs party icons | `|switch|p2a: Pikachu` |
| Using a move | Move animation + damage | `|move|p2a: Pikachu|Thunderbolt|p1a: Charizard` |
| Switching | New Pokémon enters | `|switch|p2a: Garchomp` |
| Terastallize | Type crystal overlay | `|terastallize|p2a: Pikachu|Electric` |

![Pikachu official artwork](https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/25.png)

![Charizard official artwork](https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/6.png)

![Garchomp official artwork](https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/445.png)

![Showdown battle backdrop](https://play.pokemonshowdown.com/fx/water.gif)

*Species art via [PokéAPI sprites](https://github.com/PokeAPI/sprites); animated backdrop from [Pokémon Showdown](https://pokemonshowdown.com/).*

### 3b. Next cell — interactive demo

The demo below:

1. Pulls species stats/types from the official **[PokéAPI](https://pokeapi.co/)** REST API.
2. Runs a tiny **turn-based battle** in Python (`move` / `switch`) so you can feel the decision loop.
3. Optionally connects to **[Pokémon Showdown](https://pokemonshowdown.com/)** for **live ladder matching** when `poke-env` is installed (skipped cleanly otherwise).

Work through the printed turns once; later GRPO rewards will reinforce the same action shape.


In [ ]:
import json
import urllib.request

def pokeapi_get(path: str) -> dict:
    url = f"https://pokeapi.co/api/v2/{path.strip('/')}/"
    with urllib.request.urlopen(url, timeout=20) as resp:
        return json.load(resp)


def pokemon_brief(name: str) -> dict:
    data = pokeapi_get(f"pokemon/{name.lower()}")
    types = [t["type"]["name"] for t in data["types"]]
    hp = next(s["base_stat"] for s in data["stats"] if s["stat"]["name"] == "hp")
    return {"name": data["name"], "types": types, "hp": hp}


TYPE_CHART = {
    ("electric", "water"): 2.0,
    ("electric", "ground"): 0.0,
    ("fire", "grass"): 2.0,
    ("water", "fire"): 2.0,
    ("ground", "electric"): 2.0,
}


def type_multiplier(move_type: str, defender_types: list[str]) -> float:
    m = 1.0
    for dt in defender_types:
        m *= TYPE_CHART.get((move_type, dt), 1.0)
    return m


class MiniShowdownDemo:
    def __init__(self, p2_names=("pikachu", "garchomp"), p1_names=("charizard", "blastoise")):
        self.p1 = [pokemon_brief(n) for n in p1_names]
        self.p2 = [pokemon_brief(n) for n in p2_names]
        self.p1_active, self.p2_active = 0, 0
        self.p1_hp = [p["hp"]] * len(self.p1)
        self.p2_hp = [p["hp"]] * len(self.p2)
        self.turn = 1
        self.log: list[str] = []

    def _active(self, side: str):
        mons = self.p1 if side == "p1" else self.p2
        idx = self.p1_active if side == "p1" else self.p2_active
        return mons[idx], idx

    def print_state(self):
        p1, _ = self._active("p1")
        p2, _ = self._active("p2")
        print(f"\n=== Turn {self.turn} ===")
        print(f"p1a: {p1['name']} HP={self.p1_hp[self.p1_active]} types={p1['types']}")
        print(f"p2a: {p2['name']} HP={self.p2_hp[self.p2_active]} types={p2['types']}")
        print("Bench p2:", [m["name"] for i, m in enumerate(self.p2) if i != self.p2_active])

    def apply_action(self, cmd: str) -> bool:
        cmd = cmd.strip().lower()
        if cmd.startswith("switch "):
            target = cmd.split(" ", 1)[1]
            for i, mon in enumerate(self.p2):
                if mon["name"] == target and i != self.p2_active and self.p2_hp[i] > 0:
                    self.log.append(f"|switch|p2a: {mon['name'].title()}")
                    self.p2_active = i
                    return True
            print("Illegal switch — pick a living benched species name.")
            return False
        if cmd.startswith("move "):
            move_name = cmd.split(" ", 1)[1]
            p1, _ = self._active("p1")
            p2, _ = self._active("p2")
            move_type = "electric" if "thunder" in move_name else "fire" if "flame" in move_name else p2["types"][0]
            dmg = int(20 * type_multiplier(move_type, p1["types"]))
            self.p1_hp[self.p1_active] = max(0, self.p1_hp[self.p1_active] - dmg)
            self.log.append(
                f"|move|p2a: {p2['name'].title()}|{move_name.title()}|p1a: {p1['name'].title()}"
            )
            print(f"Dealt ~{dmg} damage.")
            if self.p1_hp[self.p1_active] == 0:
                print(f"{p1['name']} fainted!")
            return True
        print("Use: move <name>  OR  switch <species>")
        return False

    def play_demo_round(self):
        self.print_state()
        print("\nTry: move thunderbolt   or   switch garchomp")
        for cmd in ("move thunderbolt", "switch garchomp", "move earthquake"):
            print(f"\n> p2 command: {cmd}")
            if not self.apply_action(cmd):
                continue
            self.turn += 1
            if self.p1_hp[self.p1_active] == 0:
                break
            self.print_state()
        print("\n--- protocol tail ---")
        for line in self.log[-4:]:
            print(line)


print("Fetching species data from PokéAPI...")
demo = MiniShowdownDemo()
demo.play_demo_round()

try:
    import asyncio
    from poke_env.player import RandomPlayer
    from poke_env.ps_client.server_configuration import ShowdownServerConfiguration

    async def _ladder_probe():
        player = RandomPlayer(
            server_configuration=ShowdownServerConfiguration,
            max_concurrent_battles=1,
            start_listening=True,
        )
        await player.ladder(1)
        await player.ps_client.stop_listening()
        return player.n_finished_battles

    print("\n[poke-env] Running 1 ladder battle on play.pokemonshowdown.com (optional)...")
    finished = asyncio.get_event_loop().run_until_complete(_ladder_probe())
    print(f"[poke-env] finished_battles={finished}")
except Exception as exc:
    print("\n[poke-env] Skipped live ladder demo (install with: pip install poke-env).")
    print(f"  reason: {exc}")


<a id="step-4-compare-weights"></a>

## 4. Compare published model weights (same battle prompt)

**Step by step:** download (or reuse cache for) two checkpoints and run **identical** inference on the tutorial demo log:

1. **Base** [`Qwen/Qwen3-4B`](https://huggingface.co/Qwen/Qwen3-4B) — general chat model, not battle-tuned.
2. **Published agent** [`GoldenGrapeGentleman1/pokemon-showdown-agent-v6`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-v6) — production-scale recipe.

Models load **one at a time** to save VRAM. Results are stored in `comparison_rows` for Step 10.


In [ ]:
import gc
import os
import sys
from pathlib import Path

import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

_root = Path(os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"]).resolve()
_eval_dir = _root / "scripts" / "eval"
sys.path.insert(0, str(_eval_dir))
from showdown_agent_eval import postprocess_agent_response, tutorial_demo_log_with_request

COMPARE_MODELS = [
    ("base", "Qwen/Qwen3-4B"),
    ("published_agent", "GoldenGrapeGentleman1/pokemon-showdown-agent-v6"),
]

SYS_MSG = (
    "You are a Pokemon Showdown battle AI. You play as p2. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)
USER_MSG = tutorial_demo_log_with_request()


def predict_action(model, tokenizer, label: str) -> str:
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": USER_MSG},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=48,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )
    raw = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1] :], skip_special_tokens=True)
    action = postprocess_agent_response(raw).splitlines()[0].strip()
    print(f"[{label}] {action}")
    return action


comparison_rows = []
for label, model_id in COMPARE_MODELS:
    print(f"\nLoading {model_id} ...")
    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=2048,
        dtype=torch.bfloat16,
        load_in_4bit=False,
        attn_implementation="sdpa",
    )
    tok = get_chat_template(tok, chat_template="qwen3")
    FastLanguageModel.for_inference(m)
    action = predict_action(m, tok, label)
    comparison_rows.append({"checkpoint": label, "model_id": model_id, "action": action})
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()

print("\n| checkpoint | action |")
print("|---|---|")
for row in comparison_rows:
    print(f"| {row['checkpoint']} | `{row['action']}` |")

print("\nStep 6 below loads the base model again for LoRA + training.")


<a id="step-5-dataset-processing"></a>

## 5. Dataset processing — from long replay to short training row

Public replays are **long** (full battle). Supervised fine-tuning needs **short** rows: log prefix up to the decision turn + one assistant action line.

### Step 5a — Inspect one raw replay


In [ ]:
from datasets import load_dataset

_preview_stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
_raw_example = next(iter(_preview_stream))
_raw_log = _raw_example["log"]
print(f"formatid={_raw_example.get('formatid')} rating={_raw_example.get('rating')}")
print(f"raw_log_chars={len(_raw_log)}  raw_log_lines={len(_raw_log.splitlines())}")
print("--- head ---")
print("\n".join(_raw_log.splitlines()[:12]))
print("--- tail ---")
print("\n".join(_raw_log.splitlines()[-8:]))


### Step 5b — Parse protocol lines and find the winner side


In [ ]:
def showdown_fields(line: str) -> list[str]:
    parts = line.strip().split("|")
    while parts and parts[0] == "":
        parts = parts[1:]
    return parts


def extract_winner_side(log_text: str):
    winner = None
    players = {}
    for line in log_text.split("\n"):
        f = showdown_fields(line)
        if len(f) >= 3 and f[0] == "player":
            players[f[1]] = f[2]
        if len(f) >= 2 and f[0] == "win":
            winner = f[1]
    if not winner:
        return None
    for side, name in players.items():
        if name == winner:
            return side
    return None

winner_side = extract_winner_side(_raw_log)
print(f"winner_side={winner_side}")


### Step 5c — Slice one decision turn (long log → short prefix)


In [ ]:
MAX_LOG_CHARS = 6000

lines = _raw_log.strip().split("\n")
turn_positions = []
for i, line in enumerate(lines):
    f = showdown_fields(line)
    if len(f) >= 2 and f[0] == "turn":
        try:
            turn_positions.append((int(f[1]), i))
        except ValueError:
            pass

print(f"turn_markers={turn_positions[:6]} ... total={len(turn_positions)}")
target_turn_idx = 0 if len(turn_positions) == 2 else 1
_, turn_line_idx = turn_positions[target_turn_idx]
next_turn_line = turn_positions[target_turn_idx + 1][1] if target_turn_idx + 1 < len(turn_positions) else len(lines)

action = None
for j in range(turn_line_idx + 1, next_turn_line):
    f = showdown_fields(lines[j])
    if len(f) < 3:
        continue
    if f[0] == "move" and f[1].startswith(f"{winner_side}a:"):
        action = f"move {f[2]}"
        break
    if f[0] == "switch" and f[1].startswith(f"{winner_side}a:"):
        pokemon = f[1].split(": ", 1)[1] if ": " in f[1] else f[1]
        action = f"switch {pokemon}"
        break

log_prefix = "\n".join(lines[: turn_line_idx + 1])
print(f"extracted_action={action}")
print(f"prefix_chars_before_cap={len(log_prefix)}")
if len(log_prefix) > MAX_LOG_CHARS:
    print(f"prefix exceeds MAX_LOG_CHARS={MAX_LOG_CHARS} — row would be dropped in training")
else:
    print("prefix fits — good training candidate")
print("--- short prefix (last 10 lines) ---")
print("\n".join(log_prefix.splitlines()[-10:]))


### Step 5d — Wrap as chat messages (what SFT/GRPO consume)


In [ ]:
SYSTEM_TEMPLATE = (
    "You are a Pokemon Showdown battle AI. You play as {side}. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)

messages = [
    {"role": "system", "content": SYSTEM_TEMPLATE.format(side=winner_side)},
    {"role": "user", "content": log_prefix},
    {"role": "assistant", "content": action},
]
print("chat messages (preview):")
for m in messages:
    preview = m["content"][:120].replace("\n", " ")
    print(f"  {m['role']}: {preview}...")


## 6. Load Qwen3-4B and attach LoRA adapters
**Tutorial content — model and adapters.** We load the open **Apache-2.0** base model [`Qwen/Qwen3-4B`](https://huggingface.co/Qwen/Qwen3-4B) and attach **LoRA** so only a small fraction of weights are trained.

For AMD ROCm tutorials we keep **`load_in_4bit=False`** for a stable, reproducible path; you can explore 4-bit later once the baseline passes review hardware. **`attn_implementation="sdpa"`** is set for efficient attention on recent PyTorch builds.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048
dtype = torch.bfloat16
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    attn_implementation="sdpa",
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"trainable_params={trainable_params / 1e6:.1f}M")
print(f"trainable_ratio={100 * trainable_params / all_params:.2f}%")

## 5e. Batch streaming — format replay logs at scale
**Tutorial content — dataset pipeline (step by step).** Steps 5a–5d walked one replay manually; this cell **automates** the same logic. Core idea: train directly from **raw** Pokemon Showdown replay logs (not hand-written battle summaries). The code uses **`streaming=True`** end to end and only materializes a **small in-memory subset** (`MAX_TRAIN_SAMPLES`) so you do not need tens of gigabytes of local parquet cache.

In [ ]:

from datasets import Dataset, load_dataset

MIN_RATING = 1400
MAX_TRAIN_SAMPLES = 2000
SHUFFLE_BUFFER = 10_000
MAX_LOG_CHARS = 6000

SYSTEM_TEMPLATE = (
    "You are a Pokemon Showdown battle AI. You play as {side}. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)

# showdown_fields, extract_winner_side from Step 5b


def format_sample(example):
    log_text = example["log"]
    side = extract_winner_side(log_text)
    if not side:
        return {"text": ""}

    lines = log_text.strip().split("
")
    turn_positions = []
    for i, line in enumerate(lines):
        f = showdown_fields(line)
        if len(f) >= 2 and f[0] == "turn":
            try:
                turn_positions.append((int(f[1]), i))
            except ValueError:
                pass

    if len(turn_positions) < 2:
        return {"text": ""}

    target_turn_idx = 0 if len(turn_positions) == 2 else 1
    _, turn_line_idx = turn_positions[target_turn_idx]
    next_turn_line = (
        turn_positions[target_turn_idx + 1][1]
        if target_turn_idx + 1 < len(turn_positions)
        else len(lines)
    )

    action = None
    for j in range(turn_line_idx + 1, next_turn_line):
        f = showdown_fields(lines[j])
        if len(f) < 3:
            continue
        if f[0] == "move" and f[1].startswith(f"{side}a:"):
            tera = ""
            start_look = max(0, j - 3)
            end_look = min(len(lines), j + 3)
            if any(
                "terastallize" in lines[k] and side in lines[k]
                for k in range(start_look, end_look)
            ):
                tera = " terastallize"
            action = f"move {f[2]}{tera}"
            break
        if f[0] == "switch" and f[1].startswith(f"{side}a:"):
            pokemon = f[1].split(": ", 1)[1] if ": " in f[1] else f[1]
            action = f"switch {pokemon}"
            break

    if not action:
        return {"text": ""}

    log_prefix = "\n".join(lines[:turn_line_idx + 1])
    if len(log_prefix) > MAX_LOG_CHARS:
        return {"text": ""}

    messages = [
        {"role": "system", "content": SYSTEM_TEMPLATE.format(side=side)},
        {"role": "user", "content": log_prefix},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


print("Streaming replay logs (same logic as Step 5, batched)...")
stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
stream = stream.shuffle(seed=3407, buffer_size=SHUFFLE_BUFFER)
stream = stream.filter(
    lambda row: "gen9" in str(row.get("formatid") or "").lower().replace(" ", "")
    and (row.get("rating") or 0) >= MIN_RATING
)

train_samples = []
scanned = 0
for row in stream:
    scanned += 1
    formatted = format_sample(row)
    if formatted["text"]:
        train_samples.append(formatted)
    if len(train_samples) >= MAX_TRAIN_SAMPLES:
        break

if not train_samples:
    raise RuntimeError(
        "No training samples were collected. Check dataset access, filters, or disk/network configuration."
    )

train_dataset = Dataset.from_list(train_samples).shuffle(seed=3407)
print(f"collected_examples={len(train_dataset)}")
print(f"replays_scanned={scanned}")
print(train_dataset[0]["text"][:1000])


## 7. Quick validation SFT
**Tutorial content — training.** This step is **not** meant to maximize ladder Elo; it is a **50-step** demonstration that the model shifts from chatty text toward **short `move` / `switch` commands**. For AMD Hub review, confirm this cell completes on your target GPU without OOM; increase `max_steps` only after you document VRAM and runtime.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

output_dir = "outputs/pokemon_showdown_agent_tutorial"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=50,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=output_dir,
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

<a id="step-8-grpo-reward"></a>

## 8. GRPO — learn battle actions with a designed reward function

**GRPO** (Group Relative Policy Optimization) samples multiple completions per prompt and nudges the policy toward higher-reward actions. Run it **after** the short SFT pass (Step 7) so the model already emits Showdown-shaped lines.

### Reward design (step by step)

| Signal | Weight | Meaning |
|--------|--------|---------|
| `structure_ok` | +2 / −3 | Single valid `move …` or `switch …` line (binary gate) |
| `move_legal_in_request` | +3 | Move listed and not disabled in `|request|` JSON |
| `switch_legal_in_request` | +3 | Switch target is a legal bench slot |
| illegal tera | −2 | `terastallize` when `canTerastallize` is false |

All checks use **`showdown_agent_eval.validate_action_against_log`** so GRPO training aligns with Step 9–10 evaluation.

**Optional packages:** some TRL builds import `mergekit` / `llm-blender` when loading `GRPOTrainer`. The next cell checks availability and skips cleanly if missing.


In [ ]:
import importlib.util
import sys
from pathlib import Path

from datasets import Dataset
from unsloth import is_bfloat16_supported

_root = Path(os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"]).resolve()
sys.path.insert(0, str(_root / "scripts" / "eval"))
from showdown_agent_eval import validate_action_against_log

grpo_trainer = None
missing_optional = [
    pkg
    for mod, pkg in [("mergekit", "mergekit"), ("llm_blender", "llm-blender")]
    if importlib.util.find_spec(mod) is None
]

if missing_optional:
    print("Skipping GRPO because optional packages are missing:")
    print(f"  pip install {' '.join(missing_optional)}")
else:
    try:
        from trl import GRPOConfig, GRPOTrainer
    except Exception as exc:
        print("Skipping GRPO — could not import GRPOTrainer.")
        print(repr(exc))
    else:
        def _log_from_prompt(prompt: str) -> str:
            marker = "<|im_start|>user\n"
            if marker in prompt:
                body = prompt.split(marker, 1)[1]
                if "<|im_start|>" in body:
                    body = body.split("<|im_start|>", 1)[0]
                return body.strip()
            return ""

        def showdown_reward_func(prompts, completions, **kwargs):
            rewards = []
            for prompt, completion in zip(prompts, completions):
                text = completion[0]["content"] if isinstance(completion, list) else str(completion)
                first_line = text.strip().splitlines()[0] if text.strip() else ""
                log_text = _log_from_prompt(prompt if isinstance(prompt, str) else str(prompt))
                checks = validate_action_against_log(first_line, log_text)

                reward = 0.0
                reward += 2.0 if checks["structure_ok"] and checks["single_line_ok"] else -3.0
                if checks["request_present"]:
                    if checks.get("move_legal_in_request") is True:
                        reward += 3.0
                    if checks.get("switch_legal_in_request") is True:
                        reward += 3.0
                    if checks.get("tera_legal_in_request") is False:
                        reward -= 2.0
                rewards.append(reward)
            return rewards

        grpo_prompts = []
        for sample in train_samples[:64]:
            prompt = sample["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"
            grpo_prompts.append({"prompt": prompt})
        grpo_dataset = Dataset.from_list(grpo_prompts)

        grpo_config = GRPOConfig(
            output_dir="outputs/pokemon_showdown_agent_grpo",
            learning_rate=3e-6,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_generations=4,
            max_completion_length=96,
            temperature=1.0,
            max_steps=10,
            logging_steps=1,
            report_to="none",
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
        )

        grpo_trainer = GRPOTrainer(
            model=model,
            reward_funcs=[showdown_reward_func],
            args=grpo_config,
            train_dataset=grpo_dataset,
        )
        print("GRPO trainer ready — uncomment grpo_trainer.train() when you want RL fine-tuning.")

# if grpo_trainer is not None:
#     grpo_trainer.train()


## 9. Inference sanity check
**Tutorial content — qualitative check.** A healthy post-SFT output should be compact and action-shaped. After only 50 steps it will not be perfect, but it should usually look like `move ...` or `switch ...` instead of free-form commentary.

This cell checks more than a single regex on the first token:

- **Line shape:** one non-empty line, matching `move ...` or `switch ...`, with an optional trailing `terastallize`.
- **Log consistency (lightweight):** we infer p2’s bench only from `|switch|p2*` lines already present in the pasted log (not from unrevealed team preview). A `switch` target must match one of those species and cannot be the current p2a. If the log has not yet shown every teammate, this check can false-negative a legal switch.
- **Turn legality (`|request|` JSON):** when the log contains a Showdown `|request|{...}` line for the current decision, we treat it as authoritative for this turn: `move` must match a non-disabled entry in `active[].moves`, `switch` must match a non-fainted, non-active Pokémon in `side.pokemon`, and `terastallize` is only allowed if `canTerastallize` is set. Many public replays omit `|request|`; paste it from a live battle or bot client when you need true legality checks.

This still does **not** cover doubles targeting, some forced-choice edge cases, or server-side PP tracking beyond what the request encodes. Treat these checks as **necessary but not sufficient** for a production ladder agent.

Module: **`showdown_agent_eval`** from the **`pokemon-rocm-tutorial`** companion repo (cloned in Step 0): `tutorial_demo_log_with_request`, `validate_action_against_log`.

In [ ]:
import os
import sys
from pathlib import Path

import torch

_root = Path(os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"]).resolve()
_eval_dir = _root / "scripts" / "eval"
if not (_eval_dir / "showdown_agent_eval.py").is_file():
    raise FileNotFoundError(
        f"Missing {_eval_dir}/showdown_agent_eval.py — re-run Step 0 (companion clone) or set POKEMON_SHOWDOWN_EVAL_ROOT."
    )
sys.path.insert(0, str(_eval_dir))

from showdown_agent_eval import (
    postprocess_agent_response,
    tutorial_demo_log_with_request,
    validate_action_against_log,
)

model.eval()
FastLanguageModel.for_inference(model)

sys_msg = (
    "You are a Pokemon Showdown battle AI. You play as p2. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)
user_msg = tutorial_demo_log_with_request()

messages = [
    {"role": "system", "content": sys_msg},
    {"role": "user", "content": user_msg},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
if "<|im_start|>assistant\n" in full_response:
    response = full_response.split("<|im_start|>assistant\n")[-1]
else:
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
response = postprocess_agent_response(response)

first_line = response.splitlines()[0].strip() if response.strip() else ""
checks = validate_action_against_log(first_line, user_msg)

print("--- AI AGENT PREDICTION ---")
print(response)
print("--- legality & format (showdown_agent_eval) ---")
print(f"structure_ok={checks['structure_ok']}")
print(f"single_line_ok={checks['single_line_ok']}")
print(f"request_present={checks['request_present']}")
if checks["move_name_nonempty"] is not None:
    print(f"move_name_nonempty={checks['move_name_nonempty']}")
if checks["switch_target_ok"] is not None:
    print(f"switch_target_ok={checks['switch_target_ok']}")
if checks["move_legal_in_request"] is not None:
    print(f"move_legal_in_request={checks['move_legal_in_request']}")
if checks["switch_legal_in_request"] is not None:
    print(f"switch_legal_in_request={checks['switch_legal_in_request']}")
if checks.get("parsed") and checks["parsed"].get("tera") and checks["tera_legal_in_request"] is not None:
    print(f"tera_legal_in_request={checks['tera_legal_in_request']}")
print(f"notes={checks['notes']}")


## 10. Mini-eval (reference protocol)

**Tutorial content — quantitative check.** After the short SFT run, we report **the same headline metrics** as the CLI helper **`eval_showdown_agent.py`** in the companion repo, implemented in **`showdown_agent_eval`**:

- **Valid format:** output contains a parsable `move …` or `switch …` (same regex as the standalone script).
- **Type match:** predicted move vs switch matches the ground-truth action type.
- **Exact match:** normalized target string matches the replay’s winner action for that turn.

This notebook uses a **small** `EVAL_SAMPLES` (streaming, held-out from training) so the loop stays fast on tutorial hardware. A fully trained reference run typically uses larger `samples` (50–200+). Expect **low exact match** after only 50 optimizer steps; the point is to compare checkpoints and to align with `eval_showdown_agent.py`, not to claim ladder strength.

Results are written next to your checkpoint under `eval_tutorial_protocol.json`.


In [ ]:
import os
import random
import sys
from pathlib import Path

import torch
from datasets import load_dataset

_root = Path(os.environ["POKEMON_SHOWDOWN_EVAL_ROOT"]).resolve()
_eval_dir = _root / "scripts" / "eval"
if not (_eval_dir / "showdown_agent_eval.py").is_file():
    raise FileNotFoundError(
        f"Missing {_eval_dir}/showdown_agent_eval.py — re-run Step 0 (companion clone) or set POKEMON_SHOWDOWN_EVAL_ROOT."
    )
sys.path.insert(0, str(_eval_dir))

from showdown_agent_eval import (
    build_test_samples,
    eval_showdown_agent_batch,
    print_metrics_summary,
    save_metrics_json,
)

EVAL_SEED = 3407
EVAL_SAMPLES = 24
random.seed(EVAL_SEED)
torch.manual_seed(EVAL_SEED)

print("Streaming held-out eval examples (same sampling logic as eval_showdown_agent.py)...")
stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
stream = stream.shuffle(seed=EVAL_SEED + 11, buffer_size=10_000)
eval_samples = build_test_samples(
    stream, EVAL_SAMPLES, seed=EVAL_SEED, max_scans=400_000
)
print(f"collected_eval_samples={len(eval_samples)}")

if not eval_samples:
    stream2 = load_dataset(
        "milkkarten/pokemon-showdown-replays-merged",
        split="train",
        streaming=True,
    ).shuffle(seed=EVAL_SEED + 99, buffer_size=10_000)
    eval_samples = build_test_samples(
        stream2,
        EVAL_SAMPLES,
        seed=EVAL_SEED,
        require_gen9=False,
        max_scans=400_000,
    )
    print(f"collected_eval_samples_relaxed_gen={len(eval_samples)}")

if not eval_samples:
    raise RuntimeError("No eval samples collected; check dataset access or filters.")

model.eval()
metrics, _rows = eval_showdown_agent_batch(
    model,
    tokenizer,
    eval_samples,
    max_new_tokens=30,
    use_cache=False,
)

print_metrics_summary(metrics, title="Tutorial SFT checkpoint — mini-eval (reference protocol)")
out_path = Path(output_dir).resolve() / "eval_tutorial_protocol.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
save_metrics_json(
    str(out_path),
    metrics,
    extra={
        "protocol": "showdown_agent_eval.eval_showdown_agent_batch",
        "eval_samples": EVAL_SAMPLES,
        "eval_seed": EVAL_SEED,
        "output_dir": output_dir,
        "trainer_max_steps": 50,
        "note": "Compare to a full checkpoint using the same metrics; increase EVAL_SAMPLES or train longer for tighter numbers.",
    },
)
print(f"Wrote metrics to {out_path}")


print("\nCompare to Step 4 baselines (same demo log):")
for row in globals().get("comparison_rows", []):
    print(f"  {row['checkpoint']}: {row['action']}")
if not globals().get("comparison_rows"):
    print("  (run Step 4 first to populate comparison_rows)")
print("  tutorial_checkpoint: see valid_format / exact_match above")


<a id="8-export-and-publish"></a>

## 11. Export and publish
**Tutorial content — artifacts.** If you want a **standalone merged** checkpoint for sharing or for Hugging Face Hub upload, export under the same `outputs/` tree as the trainer.

For **AMD AI Developer Hub** intake (`ROCm/gpuaidev-internal`), follow the project’s **branch + PR** workflow: add this `.ipynb` in the folder maintainers specify, update the repo **README** and **Sphinx** TOC as required, and note the **ROCm image + GPU SKU** you used for testing.

```python
export_dir = f"{output_dir}/merged_model"
model.save_pretrained_merged(export_dir, tokenizer, save_method="merged_16bit")
```

Suggested Hugging Face commands for a tutorial-only release:

```bash
hf auth whoami
hf repos create your-username/pokemon-showdown-agent-tutorial-demo --type model --exist-ok
hf upload-large-folder your-username/pokemon-showdown-agent-tutorial-demo outputs/pokemon_showdown_agent_tutorial/merged_model
hf upload your-username/pokemon-showdown-agent-tutorial-demo pokemon_llm_agent_unsloth_rocm_tutorial.ipynb pokemon_llm_agent_unsloth_rocm_tutorial.ipynb
```

The published reference weights remain on Hugging Face at the link in the Introduction.